## 面试问题

Context 中毒：一条错误观察污染后续决策，怎么隔离与回滚？

## 回答主线

一条脏观察进入 history 会污染后续决策。防护三层：入库校验与可信标注、来源可追溯(provenance)、事实与推测隔离。本 Notebook 让脏库存 `available=999`(低可信来源)驱动「承诺发货」，对比无 provenance（无法定位来源、无法回滚）与带 provenance（标记中毒后回滚承诺）。

## 真实案例

库存工具 bug 返回 `available=999`（真实为 0，来源 flaky_stock_api，低可信），agent 据此承诺发货。事后确认该来源不可信，需回滚承诺。数据为教学事件，不代表真实库存系统。

In [1]:
observations = [  # 定义带来源与可信度的观察序列。
    {"id": "o1", "field": "order_id", "value": "od-5", "source": "order_db", "trust": "high"},  # 高可信订单号。
    {"id": "o2", "field": "available", "value": 999, "source": "flaky_stock_api", "trust": "low"},  # 低可信且错误的库存。
]  # 结束观察序列定义。

print("观察条目:")  # 展示观察。
for ob in observations:  # 逐条打印观察。
    print("  ", ob["id"], ob["field"], "=", ob["value"], "trust", ob["trust"], "src", ob["source"])  # 展示来源与可信度。

观察条目:
   o1 order_id = od-5 trust high src order_db
   o2 available = 999 trust low src flaky_stock_api


## 基线（Baseline）

反面基线：无 provenance，动作不记录依赖来源。基于脏库存承诺发货，但事后不知道这个承诺是依据哪条观察做的。

In [2]:
def promise_shipping_naive(observations):  # 无 provenance：基于任意库存观察就承诺发货。
    for ob in observations:  # 遍历观察。
        if ob["field"] == "available" and ob["value"] > 0:  # 只要库存大于 0 就承诺。
            return {"action": "promise_shipping", "based_on": None}  # 无法记录依赖来源。
    return {"action": "backorder", "based_on": None}  # 否则缺货登记。

naive_promise = promise_shipping_naive(observations)  # 基于脏库存的承诺。
print("无 provenance 承诺:", naive_promise)  # 展示承诺发货但不知依据来自哪条观察。

无 provenance 承诺: {'action': 'promise_shipping', 'based_on': None}


## 失败案例与修正

无 provenance 时，脏数据被确认后无法定位下游承诺，污染无法回滚。修正是带 provenance：动作记录它依赖的观察 id，标记该观察中毒后沿依赖回滚。

In [3]:
def promise_shipping_traced(observations):  # 带 provenance：记录动作依赖的观察 id。
    for ob in observations:  # 遍历观察。
        if ob["field"] == "available" and ob["value"] > 0:  # 库存大于 0 承诺发货。
            return {"action": "promise_shipping", "based_on": ob["id"]}  # 记录依赖的观察 id。
    return {"action": "backorder", "based_on": None}  # 否则缺货登记。

def rollback_if_poisoned(action, poisoned_id):  # 标记某观察中毒后回滚依赖它的动作。
    if action["based_on"] == poisoned_id:  # 动作依赖被污染的观察。
        return {"action": "revoke_" + action["action"], "reason": "source_poisoned"}  # 回滚该动作。
    return action  # 未依赖则保留。

traced_promise = promise_shipping_traced(observations)  # 带来源的承诺。
print("带 provenance 承诺:", traced_promise)  # 展示承诺记录了依赖来源 o2。

带 provenance 承诺: {'action': 'promise_shipping', 'based_on': 'o2'}


In [4]:
poisoned = "o2"  # 事后确认 o2 来自不可信来源且数据错误。
rolled_back = rollback_if_poisoned(traced_promise, poisoned)  # 依据 provenance 回滚。
naive_rollback = rollback_if_poisoned(naive_promise, poisoned)  # 无 provenance 无法定位。
print("带 provenance 回滚结果:", rolled_back)  # 展示成功回滚承诺发货。
print("无 provenance 回滚结果:", naive_rollback, "(based_on 为空无法回滚)")  # 展示无法定位依赖。

带 provenance 回滚结果: {'action': 'revoke_promise_shipping', 'reason': 'source_poisoned'}
无 provenance 回滚结果: {'action': 'promise_shipping', 'based_on': None} (based_on 为空无法回滚)


## 结果解读

带 provenance 的承诺记录 `based_on=o2`，标记 o2 中毒后回滚为 `revoke_promise_shipping`；无 provenance 的承诺 `based_on=None`，无法定位、错误承诺永久留存。要点：入库即标注可信度，回滚沿 provenance 传播，不可逆动作对低可信观察设更高门槛。

In [5]:
print("带 provenance 能否回滚:", rolled_back["action"].startswith("revoke_"))  # 带来源可回滚。
print("无 provenance 能否回滚:", naive_rollback["action"].startswith("revoke_"))  # 无来源无法回滚。
print("被污染来源:", poisoned, "来自", [o["source"] for o in observations if o["id"] == poisoned])  # 展示污染来源。

带 provenance 能否回滚: True
无 provenance 能否回滚: False
被污染来源: o2 来自 ['flaky_stock_api']


In [6]:
assert traced_promise["based_on"] == "o2"  # 带 provenance 记录了依赖来源。
assert naive_promise["based_on"] is None  # 无 provenance 未记录来源。
assert rolled_back["action"] == "revoke_promise_shipping"  # 带来源可回滚被污染动作。
assert naive_rollback["action"] == "promise_shipping"  # 无来源无法回滚仍保留错误承诺。
assert rolled_back["reason"] == "source_poisoned"  # 回滚原因标注为来源中毒。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
